In [3]:
import requests
from bs4 import BeautifulSoup
import csv
import re
import os

# =========================
# CONFIG
# =========================
BASE_URL = "https://repositori.untidar.ac.id/index.php?p=show_detail&id={}"

OUTPUT_FILE = "dataset/untidar_raw.csv"

RANGE_START = 15000
RANGE_END = 22666
DRY_RUN_LIMIT = None

fields = [
    "DocID",
    "Abstrak_Raw",
    "Tahun",
    "Judul",
    "Penulis",
    "Fakultas",
    "Prodi"
]

# =========================
# HELPER
# =========================
def safe_text(soup, selector):
    el = soup.select_one(selector)
    return el.get_text(strip=True) if el else None


def extract_fakultas_prodi(soup):
    try:
        containers = soup.select("dl dd div")

        for c in containers:
            links = c.find_all("a")

            if len(links) >= 2:
                return (
                    links[0].get_text(strip=True),
                    links[1].get_text(strip=True)
                )

        return None, None

    except Exception:
        return None, None


def extract_tahun(text):
    m = re.search(r"(19|20)\d{2}", text)
    return m.group(0) if m else None


def extract_abstrak_with_br(abstrak_tag):
    """
    Simpan semua <br> menjadi token <BR>
    """

    abstrak_copy = BeautifulSoup(
        str(abstrak_tag),
        "html.parser"
    )

    for br in abstrak_copy.find_all("br"):
        br.replace_with(" <BR> ")

    text = abstrak_copy.get_text(
        " ",
        strip=True
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# =========================
# PREPARE OUTPUT
# =========================
os.makedirs(
    "dataset",
    exist_ok=True
)

# Buat file baru + header
with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f,
        delimiter="|"
    )

    writer.writerow(fields)

# =========================
# SCRAPING LOOP
# =========================
total_ids = (
    RANGE_END - RANGE_START
    if DRY_RUN_LIMIT is None
    else DRY_RUN_LIMIT
)

saved_count = 0

session = requests.Session()

session.headers.update({
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
})

for doc_id in range(
    RANGE_START,
    RANGE_START + total_ids
):

    url = BASE_URL.format(doc_id)

    try:
        r = session.get(
            url,
            timeout=10
        )

        r.raise_for_status()

    except Exception as e:

        print(
            f"[SKIP] {doc_id} "
            f"request error: {e}"
        )

        continue

    soup = BeautifulSoup(
        r.text,
        "html.parser"
    )

    # =========================
    # JUDUL
    # =========================
    judul = safe_text(
        soup,
        "blockquote h4"
    )

    # =========================
    # PENULIS
    # =========================
    penulis = safe_text(
        soup,
        "blockquote footer a"
    )

    # =========================
    # TAHUN
    # =========================
    full_text = soup.get_text(
        " ",
        strip=True
    )

    tahun = extract_tahun(
        full_text
    )

    # =========================
    # FAKULTAS + PRODI
    # =========================
    fakultas, prodi = extract_fakultas_prodi(
        soup
    )

    # =========================
    # ABSTRAK
    # =========================
    abstrak_tag = soup.select_one(
        "p.text-grey-darker"
    )

    if not abstrak_tag:

        print(
            f"[EMPTY] {doc_id}"
        )

        continue

    abstrak_raw = extract_abstrak_with_br(
        abstrak_tag
    )

    if not abstrak_raw:

        print(
            f"[EMPTY TEXT] {doc_id}"
        )

        continue

    # =========================
    # APPEND LANGSUNG KE CSV
    # =========================
    row = [
        doc_id,
        abstrak_raw,
        tahun,
        judul,
        penulis,
        fakultas,
        prodi
    ]

    try:

        with open(
            OUTPUT_FILE,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(
                f,
                delimiter="|"
            )

            writer.writerow(row)

        saved_count += 1

        print(
            f"[OK] {doc_id} "
            f"(saved={saved_count})"
        )

    except Exception as e:

        print(
            f"[WRITE ERROR] "
            f"{doc_id}: {e}"
        )

# =========================
# DONE
# =========================
print("\n====================")
print("SCRAPING FINISHED")
print(f"Saved rows : {saved_count}")
print(f"Output     : {OUTPUT_FILE}")
print("====================")

[SKIP] 15000 request error: 404 Client Error: Not Found for url: https://repositori.untidar.ac.id/index.php?p=show_detail&id=15000
[OK] 15001 (saved=1)
[OK] 15002 (saved=2)
[OK] 15003 (saved=3)
[OK] 15004 (saved=4)
[OK] 15005 (saved=5)
[OK] 15006 (saved=6)
[OK] 15007 (saved=7)
[OK] 15008 (saved=8)
[OK] 15009 (saved=9)
[OK] 15010 (saved=10)
[OK] 15011 (saved=11)
[OK] 15012 (saved=12)
[OK] 15013 (saved=13)
[OK] 15014 (saved=14)
[SKIP] 15015 request error: 404 Client Error: Not Found for url: https://repositori.untidar.ac.id/index.php?p=show_detail&id=15015
[OK] 15016 (saved=15)
[OK] 15017 (saved=16)
[OK] 15018 (saved=17)
[OK] 15019 (saved=18)
[OK] 15020 (saved=19)
[OK] 15021 (saved=20)
[OK] 15022 (saved=21)
[OK] 15023 (saved=22)
[SKIP] 15024 request error: 404 Client Error: Not Found for url: https://repositori.untidar.ac.id/index.php?p=show_detail&id=15024
[OK] 15025 (saved=23)
[OK] 15026 (saved=24)
[OK] 15027 (saved=25)
[OK] 15028 (saved=26)
[OK] 15029 (saved=27)
[OK] 15030 (saved=28)
